In [9]:
import certifi
import ssl
import geopy.geocoders

ctx = ssl.create_default_context(cafile=certifi.where())
geopy.geocoders.options.default_ssl_context = ctx

In [1]:
import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt
from geopy.geocoders import Nominatim

In [2]:
df = pd.read_csv("../data/processed/cleaned_accident_data.csv")
df.head()

,accident_id,city,state,latitude,longitude,date,time,hour,day_of_week,is_weekend,...,is_peak_hour,festival,risk_score,datetime,year,month,day,weekday,time_of_day,season
0,0,Pune,Maharashtra,18.680827,73.930388,2023-10-22,05:00:00,5,Sunday,1,...,0,Unknown,0.85,2023-10-22 05:00:00,2023,10,22,6,Morning,Post-Monsoon
1,1,Mumbai,Maharashtra,18.817732,72.790846,2023-05-21,04:00:00,4,Sunday,1,...,0,Unknown,0.10,2023-05-21 04:00:00,2023,5,21,6,Night,Summer
2,2,Mumbai,Maharashtra,19.096889,72.819424,2024-07-10,13:00:00,13,Wednesday,0,...,0,Unknown,0.45,2024-07-10 13:00:00,2024,7,10,2,Afternoon,Monsoon
3,3,Chandigarh,Punjab,30.787805,76.847507,2025-03-30,11:00:00,11,Sunday,1,...,0,Unknown,0.65,2025-03-30 11:00:00,2025,3,30,6,Morning,Summer
4,4,Chennai,Tamil Nadu,12.965155,80.283313,2024-01-25,16:00:00,16,Thursday,0,...,0,Unknown,0.10,2024-01-25 16:00:00,2024,1,25,3,Afternoon,Winter


In [10]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable
import time

geolocator = Nominatim(user_agent="accident_risk_project", timeout=10)

def geocode_road(road, city):
    try:
        location = geolocator.geocode(f"{road}, {city}, India")
        time.sleep(1)
        if location:
            return location.latitude, location.longitude
        else:
            print(f"[geocode_road] No match for: '{road}, {city}, India'")
            return None
    except (GeocoderTimedOut, GeocoderUnavailable) as e:
        print(f"[geocode_road] Service error for '{road}, {city}': {e}")
        return None
    except Exception as e:
        print(f"[geocode_road] Unexpected error for '{road}, {city}': {e}")
        return None

In [15]:
from math import radians, cos, sin, asin, sqrt

def haversine(lat1, lon1, lat2, lon2):
    # returns distance in kilometers
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371  # Earth radius in km
    return c * r

def find_nearby_accidents(df, lat, lon, radius_km=2):
    df = df.copy()
    df["distance_km"] = df.apply(
        lambda row: haversine(lat, lon, row["latitude"], row["longitude"]), axis=1
    )
    nearby = df[df["distance_km"] <= radius_km]
    return nearby

def calculate_risk_info(nearby_df):
    avg_risk = nearby_df["risk_score"].mean()

    if avg_risk >= 0.7:
        risk_level = "High"
    elif avg_risk >= 0.4:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    accident_count = len(nearby_df)
    peak_hours = nearby_df["hour"].value_counts().head(3).index.tolist()

    return {
        "risk_level": risk_level,
        "accident_count": accident_count,
        "peak_hours": peak_hours
    }

In [16]:
def predict_risk_for_road(road, city, df):
    coords = geocode_road(road, city)

    if coords is None:
        return "Location not found. Please check road and city name."

    lat, lon = coords
    nearby = find_nearby_accidents(df, lat, lon)

    if nearby.empty:
        return "We don’t currently have accident records for this area."

    result = calculate_risk_info(nearby)
    return {
        "road": road,
        "city": city,
        "risk_level": result["risk_level"],
        "accident_count": result["accident_count"],
        "peak_hours": result["peak_hours"]
    }

In [17]:
# Sanity check: confirm Nominatim itself is reachable and returning results
test = geolocator.geocode("Marine Drive, Mumbai, India")
print(test)

Marine Drive, Fort, A Ward, Mumbai Zone 1, Mumbai City District, Maharashtra, 400020, India


In [18]:
predict_risk_for_road("MG Road","Bangalore",df)

{'road': 'MG Road',
 'city': 'Bangalore',
 'risk_level': 'Medium',
 'accident_count': 16,
 'peak_hours': [19, 13, 11]}

In [19]:
predict_risk_for_road("Madhya Marg","Chandigarh",df)

{'road': 'Madhya Marg',
 'city': 'Chandigarh',
 'risk_level': 'Low',
 'accident_count': 77,
 'peak_hours': [0, 7, 4]}